# Day 2: find map locations for the 163 sections

**How to run:** in the menu choose **Runtime → Run all**. It takes about 5–8 minutes. When it finishes, your browser downloads `sections_geocoded_raw.csv`. Send that file back to Claude.

You don't need to change anything. The notebook reads your Day 1 table straight from your public GitHub repository and looks up each section on OpenStreetMap (Nominatim), one request per second as the service requires.

In [ ]:
# ==== CELL 1: load the 163 sections from your GitHub repository ====
import time, json
import pandas as pd
import requests

URL = "https://raw.githubusercontent.com/unnatisingh12/EV-charger-siting/main/sections_monthly.csv"
monthly = pd.read_csv(URL)
sections = (monthly[["section_id", "circle", "section"]]
            .drop_duplicates().sort_values("section_id").reset_index(drop=True))
print(f"Loaded {len(sections)} sections")   # should say 163

In [ ]:
# ==== CELL 2: alternative spellings to try if the official name finds nothing ====
# The official name is always tried first. These are only fallbacks, and every
# result records which search text produced it, so nothing is hidden.
ALIASES = {
    "K P H B COLONY": ["KPHB Colony"],
    "L.B.NAGAR": ["LB Nagar", "Lal Bahadur Nagar"],
    "I.D.P.L.": ["IDPL Colony", "IDPL"],
    "JEEDIMETLA(IDA)": ["IDA Jeedimetla", "Jeedimetla"],
    "MD PALLY": ["Mailardevpally", "Mylardevpally"],
    "D P PALLY": ["Dulapally"],
    "R.P. NILAYAM": ["Rashtrapati Nilayam", "Bolarum"],
    "A S RAO NAGAR": ["AS Rao Nagar", "A.S. Rao Nagar"],
    "BN REDDY NAGAR": ["BN Reddy Nagar", "B.N. Reddy Nagar"],
    "CHERLAPALLY-I": ["Cherlapally"],
    "CHERLAPALLY-II": ["Cherlapally"],
    "MEDCHAL RURAL": ["Medchal"],
    "MEDCHAL TOWN": ["Medchal"],
    "SHADNAGAR RURAL": ["Shadnagar"],
    "MOULAALI": ["Moula Ali"],
    "PAHADISHARIFF": ["Pahadi Shareef", "Pahadishareef"],
    "RAJENDERNAGAR": ["Rajendranagar"],
    "WHITE FIELD": ["Whitefields Kondapur", "Whitefields"],
    "CHUDI BAZAR": ["Chudi Bazaar"],
    "PETBASHEERABAD": ["Pet Basheerabad"],
    "UPPAL BAGAYATH": ["Uppal Bhagayath"],
    "HAYATH NAGAR": ["Hayathnagar"],
    "YELLA REDDY GUDA": ["Yellareddyguda"],
    "SRI KRISHNA NAGAR": ["Krishna Nagar Yousufguda"],
}

def search_texts(section):
    base = section.title()
    texts = [f"{base}, Hyderabad, Telangana", f"{base}, Telangana"]
    for a in ALIASES.get(section, []):
        texts += [f"{a}, Hyderabad, Telangana", f"{a}, Telangana"]
    return list(dict.fromkeys(texts))  # remove repeats, keep order

In [ ]:
# ==== CELL 3: look up every section on OpenStreetMap (takes about 5-8 minutes) ====
# Nominatim's rules: at most 1 request per second, and identify the project.
HEADERS = {"User-Agent": "EV-charger-siting research project (github.com/unnatisingh12/EV-charger-siting)"}
API = "https://nominatim.openstreetmap.org/search"
# Prefer results inside a box around greater Hyderabad, but don't exclude others,
# so that wrong-city matches can be seen and flagged rather than silently hidden.
VIEWBOX = "77.9,17.9,79.1,16.9"   # west, north, east, south

rows = []
for i, r in sections.iterrows():
    found = None
    tried = 0
    for text in search_texts(r.section):
        tried += 1
        params = {"q": text, "format": "jsonv2", "limit": 3, "countrycodes": "in",
                  "viewbox": VIEWBOX, "addressdetails": 0}
        try:
            resp = requests.get(API, params=params, headers=HEADERS, timeout=30)
            results = resp.json() if resp.status_code == 200 else []
        except Exception as e:
            results = []
        time.sleep(1.1)
        if results:
            top = results[0]
            found = {"query_used": text, "n_candidates": len(results),
                     "lat": float(top["lat"]), "lon": float(top["lon"]),
                     "osm_name": top.get("display_name"), "osm_category": top.get("category"),
                     "osm_type": top.get("type"), "osm_place_rank": top.get("place_rank"),
                     "osm_importance": top.get("importance"),
                     "other_candidates": json.dumps([{"name": c.get("display_name"),
                                                      "lat": c["lat"], "lon": c["lon"]}
                                                     for c in results[1:]])}
            break
    row = {"section_id": r.section_id, "circle": r.circle, "section": r.section,
           "queries_tried": tried, "geocode_status": "found" if found else "not_found"}
    row.update(found or {})
    rows.append(row)
    print(f"{i+1:>3}/{len(sections)}  {r.section:<25} -> {row['geocode_status']}"
          + (f"  ({found['query_used']})" if found else ""))

out = pd.DataFrame(rows)
print("\nFound:", (out.geocode_status == "found").sum(), "  Not found:", (out.geocode_status == "not_found").sum())

In [ ]:
# ==== CELL 4: save and download the result ====
out.to_csv("sections_geocoded_raw.csv", index=False)
try:
    from google.colab import files
    files.download("sections_geocoded_raw.csv")
except ImportError:
    print("Saved sections_geocoded_raw.csv")